In [ ]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything 
from cf_hill import CausalForestWrapper

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'spend' 
treatment_feature = 'treatment'

X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(float) 
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(float)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(float)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective cho Causal Forest
def objective(trial):
    # Không gian tham số tối ưu cho Causal Forest
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 52, 500, step=4), # Giữ step=4 để tránh lỗi chia hết của EconML
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 200),
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 200)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Khởi tạo mô hình qua Wrapper
        cf_model = CausalForestWrapper(
            task_type='revenue', 
            random_state=SEED, 
            **params
        )
        
        # Huấn luyện mô hình
        cf_model.fit(X=X_train, y=y_train, treatment=t_train)
        
        # Dự đoán Uplift trên tập Val
        try:
            uplift_pred_val = cf_model.predict(X_val)
        except AttributeError:
            uplift_pred_val = cf_model.model.effect(X_val).flatten()
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (Causal Forest Robust Revenue)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="CF_Robust_Revenue_Tuning", sampler=fixed_sampler)
# Có thể giảm n_trials xuống nếu chạy quá lâu
study.optimize(objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ CAUSAL FOREST BEST PARAMS TRÊN TẬP TEST")
print("="*50)

test_results = []

for SEED in seeds:
    cf_model_final = CausalForestWrapper(
        task_type='revenue',
        random_state=SEED,
        **best_params
    )
    
    cf_model_final.fit(X=X_train, y=y_train, treatment=t_train)
    
    try:
        uplift_pred_test = cf_model_final.predict(X_test)
    except AttributeError:
        uplift_pred_test = cf_model_final.model.effect(X_test).flatten()
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH CAUSAL FOREST ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu file
csv_filename = "causal_forest_revenue_robust_results.csv"
results_df.to_csv(csv_filename, index=False)

Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (Causal Forest Robust Revenue)...


  0%|          | 0/30 [00:00<?, ?it/s]

Best trial: 23. Best value: 0.635848: 100%|██████████| 30/30 [12:34<00:00, 25.16s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.6358

🚀 ĐÁNH GIÁ CAUSAL FOREST BEST PARAMS TRÊN TẬP TEST
Seed 412312: AUUC=0.275, AUQC=0.272, Lift=0.136, KRCC=-0.065
Seed 42: AUUC=0.362, AUQC=0.360, Lift=0.706, KRCC=0.038
Seed 1874: AUUC=0.309, AUQC=0.306, Lift=0.335, KRCC=-0.061
Seed 902745: AUUC=0.356, AUQC=0.353, Lift=0.208, KRCC=0.017
Seed 1: AUUC=0.397, AUQC=0.394, Lift=0.254, KRCC=0.053

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH CAUSAL FOREST (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.340 ± 0.048
Mean AUQC: 0.337 ± 0.048
Mean Lift: 0.328 ± 0.223
Mean KRCC: -0.003 ± 0.056
